# Deploying prediction model to Cognite Functions

In this example we make a simple dummy regression model that takes a single scalar input and returns a single scalar output. We schedule this model to run every minute. The input will be the latest availible data point in a CDF time series. The output will be stored in another CDF time series.

This is a minimal example. It is stronly encouraged to configure CDF objects with more descriptive metadata than is done in this example.

The following prerequisites needs to be satisified for the following example to work:
* Scikit-learn needs to be installed
* Cognite-python-sdk needs to be installed with the pandas optional dependency
* At least one time series with at least one data point with numerical data needs to be in the CDF project. Fill in the external ID of this time series below

In [ ]:
input_ts_ex_id = ""

## Initialize the Cognite client
We use a configuration file. See here for documentation on how to do this: https://cognite-sdk-python.readthedocs-hosted.com/en/latest/quickstart.html#instantiate-a-new-client-from-a-configuration-file

In [ ]:
from pathlib import Path
import yaml
from cognite.client import CogniteClient
config_path = Path("cognite-sdk-config.yaml")
config = yaml.safe_load(config_path.read_text())
client = CogniteClient.load(config)

Run the followig cell to test your connection to CDF.

In [ ]:
client.iam.token.inspect()

## Make a simple prediction model and save it to a file
Note that the fitted model has to be persisted in a file. You cannot directly deploy the fitted model as a Python object.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
n_points = 100
x = np.arange(n_points)
y = 3 * x + 2 + np.random.normal(size=100)
reg = LinearRegression().fit(x.reshape(-1, 1), y)

import pickle
with open('model.pickle', 'wb') as f:
    pickle.dump(reg, f, pickle.HIGHEST_PROTOCOL)

## Make the model output time series

In [ ]:
from cognite.client.data_classes import TimeSeriesWrite
output_ts_ex_id = "y_test_model_prediction"
ts = TimeSeriesWrite(external_id=output_ts_ex_id)
client.time_series.create(ts)

## Make the function that will run the predictions

This function needs to be called `handler` and be saved to a file called handler.py

In [ ]:
%%writefile handler.py
import pandas as pd
import pickle
import sklearn

def handle(client, data):
    with open("model.pickle", "rb") as f:
        reg = pickle.load(f)
    x = client.time_series.data.retrieve_latest(external_id=d).to_pandas()
    y = reg.predict(x)
    y = pd.DataFrame(y, index=x.index, columns=["y_test_model_prediction"])
    client.time_series.data.insert_dataframe(y, external_id_headers=True)

The deployed function needs to know what dependencies to install. These dependencies are listed in a requirements.txt file

In [ ]:
%%writefile requirements.txt
pandas
scikit-learn

All files we want to deploy needs to be put in a common folder

In [ ]:
p = Path("function/")
p.mkdir(parents=True, exist_ok=True)
for file in ["handler.py", "requirements.txt", "model.pickle"]:
    Path(file).rename(f"function/{file}")

## Deploy the function and make a schedule to run in every minute

In [ ]:
name = "Test_prediction_model"
function = client.functions.create(name=name, folder="function")

In [ ]:
cron_expression = "* * * * *"  # Run prediction every minute
data = {"input_ts_ex_id": }
schedule = client.functions.schedules.create("Test_prediction_model_schedule", 
                                             cron_expression=cron_expression, 
                                             function_id=function.id, 
                                             data=data)

## Clean up

In [ ]:
client.functions.schedules.delete(schedule.id)

In [ ]:
client.functions.delete(function.id)

In [ ]:
import shutil
shutil.rmtree(p)

In [ ]:
client.time_series.delete(external_id=output_ts_ex_id)